In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
from pathlib import Path
from datasets import load_dataset

# ------------------------------------------------------------------
# Load the PolitikWeli dataset
# ------------------------------------------------------------------

DATA_DIR = Path("data")
POLITIKWELI_DIR = DATA_DIR / "kweli-main"

POLITIKWELI_FILES = {
    "swahili_english": POLITIKWELI_DIR / "swa-eng.csv",
    "swahili_only": POLITIKWELI_DIR / "swa.csv",
}

# Verify that all required files are available
missing = [
    path for path in POLITIKWELI_FILES.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "The following PolitikWeli dataset files were not found:\n"
        + "\n".join(f" - {path}" for path in missing)
    )

politikweli_dataset = load_dataset(
    "csv",
    data_files={
        split: str(path)
        for split, path in POLITIKWELI_FILES.items()
    },
)

print("✓ PolitikWeli dataset loaded successfully.")
print(politikweli_dataset)

In [ ]:
# ==========================================================
# 01_data_preparation.ipynb
# Section 1: Configuration & Text Preprocessing
# ==========================================================
#
# Purpose
# -------
# Prepare the hydrated PolitikWeli datasets for downstream
# harmonisation and model training.
#
# This section:
#   • Loads the hydrated datasets
#   • Cleans tweet text
#   • Removes invalid records
#   • Analyses sequence lengths
#
# Outputs (held in memory)
# ------------------------
# swa_df
# swa_eng_df
#
# ==========================================================

from pathlib import Path
import logging
import re

import numpy as np
import pandas as pd

# ==========================================================
# Configuration
# ==========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

RANDOM_STATE = 42

PROJECT_ROOT = Path(".")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "politikweli"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "swahili": {
        "name": "Swahili",
        "input": RAW_DIR / "hydrated_valid_swa_master.csv",
    },
    "codeswitched": {
        "name": "Swahili-English (Code-Switched)",
        "input": RAW_DIR / "hydrated_valid_swa_eng_master.csv",
    },
}

# ==========================================================
# Regular Expressions
# ==========================================================

ZERO_WIDTH_PATTERN = re.compile(r'[\u200b\u200c\u200d\ufeff]')
RETWEET_PATTERN = re.compile(r'^RT\s+@\w+:\s*', flags=re.IGNORECASE)
URL_PATTERN = re.compile(r'http\S+|www\S+|https\S+')
MENTION_PATTERN = re.compile(r'@\w+')
WHITESPACE_PATTERN = re.compile(r'\s+')

# ==========================================================
# Text Cleaning
# ==========================================================

def clean_tweet_text(text: str) -> str:
    """
    Clean tweet text while preserving:
        - hashtags
        - emojis
        - code-switching
        - slang

    The cleaning procedure follows the preprocessing
    pipeline used in the study.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    text = ZERO_WIDTH_PATTERN.sub("", text)

    text = RETWEET_PATTERN.sub("", text)

    text = URL_PATTERN.sub("", text)

    text = MENTION_PATTERN.sub("@USER", text)

    text = WHITESPACE_PATTERN.sub(" ", text).strip()

    return text


# ==========================================================
# Dataset Loading
# ==========================================================

def load_dataset(csv_path: Path) -> pd.DataFrame:
    """
    Load a hydrated PolitikWeli dataset.
    """

    if not csv_path.exists():
        raise FileNotFoundError(
            f"Dataset not found:\n{csv_path}"
        )

    df = pd.read_csv(
        csv_path,
        dtype={"tweet id": str}
    )

    logger.info(
        f"Loaded {len(df):,} records from {csv_path.name}"
    )

    return df


# ==========================================================
# Preprocessing
# ==========================================================

def preprocess_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove invalid rows and clean tweet text.
    """

    original_rows = len(df)

    # Remove missing labels
    df = df.dropna(subset=["label"]).copy()

    removed = original_rows - len(df)

    if removed > 0:
        logger.info(
            f"Removed {removed:,} rows with missing labels."
        )

    # Clean text
    df["clean_text"] = df["full_text"].apply(
        clean_tweet_text
    )

    # Remove empty text
    before = len(df)

    df = df[df["clean_text"] != ""].copy()

    logger.info(
        f"Removed {before-len(df):,} empty tweets."
    )

    logger.info(
        f"Remaining records: {len(df):,}"
    )

    return df


# ==========================================================
# Sequence Length Analysis
# ==========================================================

def analyse_sequence_lengths(
    df: pd.DataFrame,
    dataset_name: str
) -> int:
    """
    Analyse cleaned tweet lengths and recommend
    an appropriate tokenizer max_length.
    """

    logger.info(
        f"Sequence length analysis: {dataset_name}"
    )

    word_counts = (
        df["clean_text"]
        .str.split()
        .str.len()
    )

    print("\n")
    print(word_counts.describe().round(2))

    p90 = np.percentile(word_counts, 90)
    p95 = np.percentile(word_counts, 95)
    p99 = np.percentile(word_counts, 99)

    logger.info(f"90th percentile : {int(p90)} words")
    logger.info(f"95th percentile : {int(p95)} words")
    logger.info(f"99th percentile : {int(p99)} words")

    recommendation = int(p95 * 1.5)

    if recommendation <= 64:
        max_length = 64
    elif recommendation <= 128:
        max_length = 128
    else:
        max_length = 256

    logger.info(
        f"Recommended tokenizer max_length = {max_length}"
    )

    return max_length


# ==========================================================
# Process Both PolitikWeli Datasets
# ==========================================================

logger.info("=" * 70)
logger.info("POLITIKWELI TEXT PREPROCESSING")
logger.info("=" * 70)

swa_df = preprocess_dataset(
    load_dataset(DATASETS["swahili"]["input"])
)

swa_eng_df = preprocess_dataset(
    load_dataset(DATASETS["codeswitched"]["input"])
)

analyse_sequence_lengths(
    swa_df,
    "Swahili"
)

analyse_sequence_lengths(
    swa_eng_df,
    "Swahili-English"
)

logger.info("Text preprocessing completed successfully.")

In [ ]:
# ==========================================================
# Section 2: Deduplication & Dataset Harmonisation
# ==========================================================
#
# Purpose
# -------
# • Remove conflicting labels
# • Remove duplicate tweets
# • Harmonise schemas
# • Encode labels
# • Merge both PolitikWeli subsets
#
# Output (held in memory)
# -----------------------
# politikweli_master
#
# ==========================================================

# ==========================================================
# Label Mapping
# ==========================================================

LABEL_MAP = {
    "neutral": 0,
    "fact": 1,
    "fake": 2
}

# ==========================================================
# Deduplication
# ==========================================================

def deduplicate_dataset(
    df: pd.DataFrame,
    text_col: str = "clean_text",
    label_col: str = "label"
) -> pd.DataFrame:
    """
    Remove:
      1. Tweets with conflicting labels.
      2. Exact duplicate tweet/label pairs.
    """

    df = df.dropna(subset=[text_col, label_col]).copy()

    initial_rows = len(df)

    # ---------------------------------------------
    # Remove conflicting labels
    # ---------------------------------------------

    label_counts = (
        df.groupby(text_col)[label_col]
          .nunique()
    )

    conflicting_texts = label_counts[
        label_counts > 1
    ].index

    df = df[
        ~df[text_col].isin(conflicting_texts)
    ].copy()

    conflicts_removed = initial_rows - len(df)

    # ---------------------------------------------
    # Remove exact duplicates
    # ---------------------------------------------

    before = len(df)

    df = df.drop_duplicates(
        subset=[text_col, label_col],
        keep="first"
    )

    duplicates_removed = before - len(df)

    logger.info(
        f"Conflicting rows removed : {conflicts_removed:,}"
    )

    logger.info(
        f"Duplicate rows removed   : {duplicates_removed:,}"
    )

    logger.info(
        f"Remaining rows           : {len(df):,}"
    )

    return df


# ==========================================================
# Harmonisation
# ==========================================================

def harmonise_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardise the PolitikWeli schema.
    """

    return df.rename(
        columns={
            "tweet id": "text_id",
            "clean_text": "text",
            "full_text": "raw_text",
            "label": "label_text"
        }
    )


def encode_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert textual labels into integer IDs.
    """

    df["label_text"] = (
        df["label_text"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    df["label"] = df["label_text"].map(LABEL_MAP)

    if df["label"].isna().any():

        invalid = (
            df.loc[df["label"].isna(), "label_text"]
            .unique()
        )

        raise ValueError(
            f"Unknown labels detected: {invalid}"
        )

    return df


def final_merge_cleanup(df: pd.DataFrame) -> pd.DataFrame:
    """
    Final duplicate/conflict check after merging.
    """

    label_counts = (
        df.groupby("text")["label_text"]
          .nunique()
    )

    conflicting = label_counts[
        label_counts > 1
    ].index

    if len(conflicting):

        logger.info(
            f"Removing {len(conflicting):,} conflicting merged texts."
        )

        df = df[
            ~df["text"].isin(conflicting)
        ].copy()

    before = len(df)

    df = df.drop_duplicates(
        subset=["text", "label_text"]
    )

    logger.info(
        f"Final duplicate rows removed: {before-len(df):,}"
    )

    return df


# ==========================================================
# Process Individual Datasets
# ==========================================================

logger.info("=" * 70)
logger.info("DEDUPLICATING POLITIKWELI SUBSETS")
logger.info("=" * 70)

# Swahili
logger.info("Swahili")

swa_df = deduplicate_dataset(swa_df)

swa_df["subset"] = "swa_only"

# Code-switched
logger.info("Swahili-English")

swa_eng_df = deduplicate_dataset(swa_eng_df)

swa_eng_df["subset"] = "swa_eng_codeswitched"

# ==========================================================
# Merge Datasets
# ==========================================================

logger.info("=" * 70)
logger.info("HARMONISING DATASETS")
logger.info("=" * 70)

politikweli_master = pd.concat(
    [
        swa_df,
        swa_eng_df
    ],
    ignore_index=True
)

politikweli_master = harmonise_columns(
    politikweli_master
)

politikweli_master = encode_labels(
    politikweli_master
)

politikweli_master = final_merge_cleanup(
    politikweli_master
)

# ==========================================================
# Keep Only Required Columns
# ==========================================================

preferred_columns = [

    "text_id",

    "text",

    "label",

    "label_text",

    "subset",

    "raw_text",

    "created_at",

    "user_screen_name",

    "likes",

    "retweets",

    "replies",

]

existing_columns = [
    column
    for column in preferred_columns
    if column in politikweli_master.columns
]

politikweli_master = politikweli_master[
    existing_columns
].copy()

logger.info("=" * 70)
logger.info("HARMONISATION COMPLETE")
logger.info("=" * 70)

logger.info(
    f"Master dataset size: {len(politikweli_master):,}"
)

logger.info("\nDistribution by subset:")

print(
    pd.crosstab(
        politikweli_master["subset"],
        politikweli_master["label_text"],
        margins=True
    )
)

print("\nMaster Dataset Preview")

display(
    politikweli_master.head()
)

In [ ]:
# ==========================================================
# Section 3: Dataset Audit
# ==========================================================
#
# Purpose
# -------
# Perform quality assurance checks on the harmonised
# PolitikWeli dataset before creating the final
# train/validation/test splits.
#
# This section does NOT modify the dataset.
#
# ==========================================================

logger.info("=" * 70)
logger.info("POLITIKWELI DATASET AUDIT")
logger.info("=" * 70)

# ==========================================================
# Missing Values
# ==========================================================

logger.info("Checking missing values...")

required_columns = [
    "text",
    "label",
    "label_text",
    "subset"
]

missing_summary = (
    politikweli_master[required_columns]
    .isnull()
    .sum()
)

print("\nMissing Values")
print("----------------")
print(missing_summary)

# ==========================================================
# Duplicate Text Audit
# ==========================================================

duplicate_texts = (
    politikweli_master["text"]
    .duplicated()
    .sum()
)

print("\nDuplicate Texts")
print("----------------")
print(f"{duplicate_texts:,}")

if duplicate_texts == 0:
    logger.info("No duplicate texts detected.")
else:
    logger.warning(
        f"{duplicate_texts:,} duplicate texts detected."
    )

# ==========================================================
# Overall Class Distribution
# ==========================================================

logger.info("Computing class distribution...")

class_distribution = pd.DataFrame({

    "Count":
        politikweli_master["label_text"]
        .value_counts(),

    "Percentage (%)":
        (
            politikweli_master["label_text"]
            .value_counts(normalize=True)
            * 100
        ).round(2)

})

print("\nOverall Class Distribution")
print("--------------------------")
print(class_distribution)

# ==========================================================
# Language Subset Distribution
# ==========================================================

subset_distribution = pd.crosstab(

    politikweli_master["subset"],

    politikweli_master["label_text"],

    margins=True

)

print("\nDistribution by Language Subset")
print("--------------------------------")
print(subset_distribution)

# ==========================================================
# Composite Stratification Key
# ==========================================================

logger.info("Creating composite stratification key...")

politikweli_master["composite_key"] = (

    politikweli_master["subset"]

    + "_"

    + politikweli_master["label_text"]

)

composite_counts = (
    politikweli_master["composite_key"]
    .value_counts()
)

print("\nComposite Groups")
print("----------------")
print(composite_counts)

smallest_group = composite_counts.idxmin()

print("\nSmallest Composite Group")
print("------------------------")
print(
    f"{smallest_group} "
    f"({composite_counts.min():,} samples)"
)

# ==========================================================
# Composite Group Summary
# ==========================================================

audit_summary = pd.DataFrame({

    "Samples":
        composite_counts,

    "Percentage (%)":
        (
            composite_counts
            / len(politikweli_master)
            * 100
        ).round(2)

})

print("\nComposite Group Summary")
print("-----------------------")
print(audit_summary)

# ==========================================================
# Audit Summary
# ==========================================================

logger.info("=" * 70)
logger.info("AUDIT SUMMARY")
logger.info("=" * 70)

print(f"\nTotal Samples : {len(politikweli_master):,}")

print(
    f"Unique Texts  : "
    f"{politikweli_master['text'].nunique():,}"
)

print(
    f"Duplicate Texts : "
    f"{duplicate_texts:,}"
)

print(
    f"Language Subsets : "
    f"{politikweli_master['subset'].nunique()}"
)

print(
    f"Classes : "
    f"{politikweli_master['label_text'].nunique()}"
)

print(
    f"Composite Groups : "
    f"{len(composite_counts)}"
)

if duplicate_texts == 0:
    logger.info("✓ Dataset passed duplicate audit.")

if missing_summary.sum() == 0:
    logger.info("✓ Dataset passed missing value audit.")

logger.info(
    "✓ Dataset audit completed successfully."
)

# ==========================================================
# Dataset Preview
# ==========================================================

print("\nDataset Preview")
print("----------------")

display(
    politikweli_master.head()
)

In [ ]:
# ==========================================================
# Section 4: Create Frozen Train / Validation / Test Splits
# ==========================================================
#
# Purpose
# -------
# Create the final frozen train, validation and test datasets
# used throughout the remainder of the study.
#
# Workflow
# --------
# 1. Create the binary misinformation target.
# 2. Attempt composite stratification (audit only).
# 3. Fall back to label stratification if composite
#    stratification is not feasible.
# 4. Generate reproducible frozen dataset splits.
#
# Outputs (Part A)
# ----------------
# • politikweli_train_df
# • politikweli_validation_df
# • politikweli_test_df
#
# Part B will perform:
# • leakage checks
# • distribution verification
# • class-weight computation
# • dataset export
# • metadata export
#
# ==========================================================

from sklearn.model_selection import train_test_split

logger.info("=" * 70)
logger.info("CREATING FROZEN TRAIN / VALIDATION / TEST SPLITS")
logger.info("=" * 70)

# ==========================================================
# 4.1 Create Binary Misinformation Target
# ==========================================================
#
# Binary target used by the multi-task model.
#
# Original labels
# ---------------
# neutral -> 0
# fact    -> 1
# fake    -> 2
#
# Binary target
# -------------
# neutral -> 0
# fact/fake -> 1
#
# ==========================================================

logger.info("Creating binary misinformation target...")

politikweli_master["is_misinfo"] = (
    politikweli_master["label"] != 0
).astype(int)

logger.info(
    "✓ Binary misinformation target successfully created."
)

# ==========================================================
# 4.2 Attempt Composite Stratification (Audit)
# ==========================================================
#
# Composite stratification preserves both:
#
# • Language subset
# • Original misinformation label
#
# This was evaluated during dataset auditing.
# If any composite group is too small to support
# three-way splitting, the notebook automatically
# falls back to stratification using the original
# three-class misinformation label.
#
# This reproduces the methodology used in the
# completed study.
#
# ==========================================================

logger.info("=" * 70)
logger.info("EVALUATING STRATIFICATION STRATEGY")
logger.info("=" * 70)

politikweli_master["composite_key"] = (

    politikweli_master["subset"].astype(str)
    + "_"
    + politikweli_master["label_text"].astype(str)

)

use_composite = True

try:

    train_test_split(

        politikweli_master,

        test_size=0.20,

        random_state=RANDOM_STATE,

        stratify=politikweli_master["composite_key"]

    )

    logger.info(
        "✓ Composite stratification is feasible."
    )

except ValueError as e:

    use_composite = False

    logger.warning(
        "Composite stratification not feasible."
    )

    logger.warning(
        f"Reason: {e}"
    )

    logger.info(
        "Proceeding with label-stratified splitting "
        "to reproduce the experimental setup."
    )

# ==========================================================
# 4.3 Create Frozen Dataset Splits
# ==========================================================

logger.info("=" * 70)
logger.info("CREATING FROZEN DATASET SPLITS")
logger.info("=" * 70)

if use_composite:

    logger.info(
        "Using composite stratification."
    )

    politikweli_train_df, temp_df = train_test_split(

        politikweli_master,

        test_size=0.20,

        random_state=RANDOM_STATE,

        stratify=politikweli_master["composite_key"]

    )

    politikweli_validation_df, politikweli_test_df = train_test_split(

        temp_df,

        test_size=0.50,

        random_state=RANDOM_STATE,

        stratify=temp_df["composite_key"]

    )

    stratification_method = "composite_key"

else:

    logger.info(
        "Using original three-class label stratification."
    )

    politikweli_train_df, temp_df = train_test_split(

        politikweli_master,

        test_size=0.20,

        random_state=RANDOM_STATE,

        stratify=politikweli_master["label"]

    )

    politikweli_validation_df, politikweli_test_df = train_test_split(

        temp_df,

        test_size=0.50,

        random_state=RANDOM_STATE,

        stratify=temp_df["label"]

    )

    stratification_method = "label"

logger.info(
    f"Stratification method: {stratification_method}"
)

# ==========================================================
# 4.4 Verify Split Sizes
# ==========================================================

total_records = len(politikweli_master)

assert (

    len(politikweli_train_df)
    + len(politikweli_validation_df)
    + len(politikweli_test_df)

) == total_records

logger.info("✓ Split integrity verified.")

logger.info("")

logger.info(
    f"Training samples:    {len(politikweli_train_df):,} "
    f"({len(politikweli_train_df)/total_records:.1%})"
)

logger.info(
    f"Validation samples: {len(politikweli_validation_df):,} "
    f"({len(politikweli_validation_df)/total_records:.1%})"
)

logger.info(
    f"Test samples:       {len(politikweli_test_df):,} "
    f"({len(politikweli_test_df)/total_records:.1%})"
)

# ==========================================================
# 4.5 Remove Temporary Audit Column
# ==========================================================

for dataframe in [

    politikweli_master,

    politikweli_train_df,

    politikweli_validation_df,

    politikweli_test_df

]:

    dataframe.drop(

        columns=["composite_key"],

        inplace=True,

        errors="ignore"

    )

logger.info(
    "✓ Temporary audit columns removed."
)

logger.info("=" * 70)
logger.info("PART A COMPLETE")
logger.info("=" * 70)
logger.info(
    "Proceed to Part B for integrity checks, "
    "dataset export and metadata generation."
)
# ==========================================================
# Section 4B: Verify Splits, Export Datasets & Metadata
# ==========================================================
#
# Purpose
# -------
# Complete the frozen dataset preparation by:
#
# • verifying data integrity
# • checking for data leakage
# • validating class distributions
# • computing binary class weights
# • exporting datasets
# • exporting metadata
#
# ==========================================================

import json
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

logger.info("=" * 70)
logger.info("VERIFYING FROZEN DATASETS")
logger.info("=" * 70)

# ==========================================================
# 4.6 Data Integrity Checks
# ==========================================================

logger.info("Running dataset integrity checks...")

assert (
    len(politikweli_train_df)
    + len(politikweli_validation_df)
    + len(politikweli_test_df)
) == len(politikweli_master)

assert politikweli_train_df.index.is_unique
assert politikweli_validation_df.index.is_unique
assert politikweli_test_df.index.is_unique

logger.info("✓ Dataset sizes verified.")
logger.info("✓ Row indices verified.")

# ==========================================================
# 4.7 Data Leakage Checks
# ==========================================================

logger.info("=" * 70)
logger.info("CHECKING FOR DATA LEAKAGE")
logger.info("=" * 70)

# ----------------------------------------------------------
# ID leakage
# ----------------------------------------------------------

if "text_id" in politikweli_train_df.columns:

    train_ids = set(politikweli_train_df["text_id"])
    validation_ids = set(politikweli_validation_df["text_id"])
    test_ids = set(politikweli_test_df["text_id"])

    assert len(train_ids & validation_ids) == 0
    assert len(train_ids & test_ids) == 0
    assert len(validation_ids & test_ids) == 0

    logger.info("✓ No ID leakage detected.")

# ----------------------------------------------------------
# Text leakage
# ----------------------------------------------------------

train_texts = set(politikweli_train_df["text"])

validation_texts = set(politikweli_validation_df["text"])

test_texts = set(politikweli_test_df["text"])

assert len(train_texts & validation_texts) == 0
assert len(train_texts & test_texts) == 0
assert len(validation_texts & test_texts) == 0

logger.info("✓ No text leakage detected.")

# ==========================================================
# 4.8 Verify Label Distributions
# ==========================================================

logger.info("=" * 70)
logger.info("VERIFYING CLASS DISTRIBUTIONS")
logger.info("=" * 70)

distribution = pd.DataFrame({

    "Overall %":
        politikweli_master["label_text"]
        .value_counts(normalize=True)
        * 100,

    "Train %":
        politikweli_train_df["label_text"]
        .value_counts(normalize=True)
        * 100,

    "Validation %":
        politikweli_validation_df["label_text"]
        .value_counts(normalize=True)
        * 100,

    "Test %":
        politikweli_test_df["label_text"]
        .value_counts(normalize=True)
        * 100,

}).round(2)

print("\nLabel Distribution (%)")
print(distribution)

logger.info("✓ Label distributions verified.")

# ----------------------------------------------------------
# Binary misinformation distribution
# ----------------------------------------------------------

binary_distribution = pd.DataFrame({

    "Overall %":
        politikweli_master["is_misinfo"]
        .value_counts(normalize=True)
        * 100,

    "Train %":
        politikweli_train_df["is_misinfo"]
        .value_counts(normalize=True)
        * 100,

    "Validation %":
        politikweli_validation_df["is_misinfo"]
        .value_counts(normalize=True)
        * 100,

    "Test %":
        politikweli_test_df["is_misinfo"]
        .value_counts(normalize=True)
        * 100,

}).round(2)

print("\nBinary Misinformation Distribution (%)")
print(binary_distribution)

# ----------------------------------------------------------
# Language subset distribution
# ----------------------------------------------------------

subset_distribution = pd.DataFrame({

    "Overall %":
        politikweli_master["subset"]
        .value_counts(normalize=True)
        * 100,

    "Train %":
        politikweli_train_df["subset"]
        .value_counts(normalize=True)
        * 100,

    "Validation %":
        politikweli_validation_df["subset"]
        .value_counts(normalize=True)
        * 100,

    "Test %":
        politikweli_test_df["subset"]
        .value_counts(normalize=True)
        * 100,

}).round(2)

print("\nLanguage Subset Distribution (%)")
print(subset_distribution)

# ==========================================================
# 4.9 Compute Binary Class Weights
# ==========================================================

logger.info("=" * 70)
logger.info("COMPUTING CLASS WEIGHTS")
logger.info("=" * 70)

weights = compute_class_weight(

    class_weight="balanced",

    classes=np.array([0, 1]),

    y=politikweli_train_df["is_misinfo"]

)

class_weights = {

    "0": float(weights[0]),

    "1": float(weights[1])

}

logger.info(
    f"Class Weights: {class_weights}"
)

# ==========================================================
# 4.10 Export Processed Datasets
# ==========================================================

logger.info("=" * 70)
logger.info("EXPORTING DATASETS")
logger.info("=" * 70)

politikweli_master.to_csv(
    PROCESSED_DIR / "politikweli_master.csv",
    index=False
)

politikweli_train_df.to_csv(
    PROCESSED_DIR / "politikweli_train.csv",
    index=False
)

politikweli_validation_df.to_csv(
    PROCESSED_DIR / "politikweli_validation.csv",
    index=False
)

politikweli_test_df.to_csv(
    PROCESSED_DIR / "politikweli_test.csv",
    index=False
)

logger.info("✓ Processed datasets exported.")

# ==========================================================
# 4.11 Export Split Summary
# ==========================================================

split_summary = pd.DataFrame({

    "Split": [

        "Train",

        "Validation",

        "Test"

    ],

    "Samples": [

        len(politikweli_train_df),

        len(politikweli_validation_df),

        len(politikweli_test_df)

    ],

    "Percentage": [

        round(len(politikweli_train_df) / len(politikweli_master) * 100, 2),

        round(len(politikweli_validation_df) / len(politikweli_master) * 100, 2),

        round(len(politikweli_test_df) / len(politikweli_master) * 100, 2)

    ]

})

split_summary.to_csv(

    PROCESSED_DIR /
    "politikweli_split_summary.csv",

    index=False

)

logger.info("✓ Split summary exported.")

# ==========================================================
# 4.12 Export Metadata
# ==========================================================

metadata = {

    "dataset": "PolitikWeli",

    "version": "1.0",

    "random_state": RANDOM_STATE,

    "stratification": stratification_method,

    "train_ratio": 0.80,

    "validation_ratio": 0.10,

    "test_ratio": 0.10,

    "total_samples": len(politikweli_master),

    "train_samples": len(politikweli_train_df),

    "validation_samples": len(politikweli_validation_df),

    "test_samples": len(politikweli_test_df),

    "label_mapping": {

        "0": "neutral",

        "1": "fact",

        "2": "fake"

    },

    "binary_target": {

        "0": "neutral",

        "1": "fact_or_fake"

    },

    "class_weights": class_weights

}

with open(

    PROCESSED_DIR /
    "politikweli_metadata.json",

    "w",

    encoding="utf-8"

) as fp:

    json.dump(
        metadata,
        fp,
        indent=4
    )

logger.info("✓ Metadata exported.")

# ==========================================================
# 4.13 Export Binary Class Weights
# ==========================================================

with open(

    PROCESSED_DIR /
    "politikweli_binary_class_weights.json",

    "w",

    encoding="utf-8"

) as fp:

    json.dump(
        class_weights,
        fp,
        indent=4
    )

logger.info("✓ Class weights exported.")

# ==========================================================
# 4.14 Completion Summary
# ==========================================================

logger.info("=" * 70)
logger.info("POLITIKWELI DATA PREPARATION COMPLETE")
logger.info("=" * 70)

print(
"""
============================================================
POLITIKWELI DATA PREPARATION SUMMARY
============================================================

✓ Raw hydrated datasets loaded
✓ Text preprocessing completed
✓ Duplicate and conflicting records removed
✓ Datasets harmonised
✓ Dataset audit completed
✓ Binary misinformation target created
✓ Frozen train/validation/test datasets created
✓ Data leakage checks passed
✓ Label distributions verified
✓ Binary class weights computed
✓ Processed datasets exported
✓ Metadata exported

Generated files
---------------

data/processed/
    politikweli_master.csv
    politikweli_train.csv
    politikweli_validation.csv
    politikweli_test.csv
    politikweli_split_summary.csv
    politikweli_metadata.json
    politikweli_binary_class_weights.json

Repository ready for AfriHate dataset preparation.
============================================================
"""
)